In [ ]:
import os
import sys
import shutil
sys.path.append('./')
path = sys.path

proc_datapath = str(path[-1:][0])+"../datasets/preprocessed/"

In [ ]:
import os
import torch
import random
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import font_manager

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

def set_random_seed(seed=42):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [ ]:
def read_preprocessed_data(data_id, datapath):
    X_tr = pd.read_csv(datapath+data_id+'_Xtrain.csv')
    X_ts = pd.read_csv(datapath+data_id+'_Xtest.csv')
    y_tr = pd.read_csv(datapath+data_id+'_ytrain.csv')
    y_ts = pd.read_csv(datapath+data_id+'_ytest.csv')
    
    return X_tr, X_ts, y_tr, y_ts

In [ ]:
class WBCELossx(nn.Module):
    def __init__(self, 
                 false_positive_weight=2.0, 
                 false_negative_weight=1.33,
                 gamma=0.01):
        super(WBCELossx, self).__init__()
        self.fp_weight = false_positive_weight
        self.fn_weight = false_negative_weight

    def forward(self, predictions, targets):
        predictions = torch.clamp(predictions, 1e-6, 1 - 1e-6)
        false_positive_loss = self.fp_weight * (1 - targets) * torch.log(1 - predictions)
        false_negative_loss = self.fn_weight * targets * torch.log(predictions)
        loss = -torch.mean(false_positive_loss + false_negative_loss)
        return loss

    
class BCELossx(nn.Module):
    def __init__(self):
        super(BCELossx, self).__init__()

    def forward(self, predictions, targets):
        predictions = torch.clamp(predictions, 1e-6, 1 - 1e-6)
        false_positive_loss = (1 - targets) * torch.log(1 - predictions)
        false_negative_loss = targets * torch.log(predictions)
        loss = -torch.mean(false_positive_loss + false_negative_loss)
        return loss
    
    
class NNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super(NNClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 1)  
        self.sigmoid = nn.Sigmoid()        
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        x = self.sigmoid(x) 
        return x

    

In [ ]:
def pgd_improvement(hfunck, loss_funck, inputsk, labelsk, sel_feat_idsk, epsilonbk, alpha_valk, num_itersk):
    
    adv_inputsk = inputsk.clone().detach().requires_grad_(True)
    ori_inputsk = inputsk.clone().detach()
    
    for _ in range(num_itersk):
        outputsk = hfunck(adv_inputsk)
        ori_outputk = hfunck(ori_inputsk)
        
        hfunck.zero_grad()

        costk = loss_funck(outputsk, ori_outputk)
        costk.backward()

        gradk = adv_inputsk.grad.clone()
        perturbationk = torch.zeros_like(adv_inputsk)
        perturbationk[:, sel_feat_idsk] = alpha_valk * gradk[:, sel_feat_idsk].sign()

        adv_inputsk = adv_inputsk.detach() + perturbationk
        adv_inputsk = torch.clamp(adv_inputsk, ori_inputsk - epsilonbk, ori_inputsk + epsilonbk)
        adv_inputsk.requires_grad_(True)

    return adv_inputsk



In [ ]:
def compute_fnr_fpr(yh, predh):
    
    cm = confusion_matrix(yh, predh, labels=[0, 1])
    misclassified = (yh != predh)
    misclassified = np.sum(misclassified)/len(yh)
    FN = (yh == 1) & (predh == 0) 
    FP = (yh == 0) & (predh == 1)

    if cm.shape == (2, 2):
        TN, FP, FN, TP = cm.ravel()
        fnr = FN / (FN + TP) if (FN + TP) > 0 else 0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
        return fnr, fpr
    else:
        fnr_error = np.sum(FN) / np.sum(yh == 1) if np.sum(yh == 1)>0 else 0 ##fn/p = fn/(tp+fn)
        fpr_error = np.sum(FP) / np.sum(yh == 0) if np.sum(yh == 0)>0 else 0 ##fp/n = fp/(tn+fp)
        print(fnr_error, fpr_error)
        return 0, 0



def calculate_error_rate(hfuncmodel, fstarx, X_dset, hthresh):
    
    y_fstar = fstarx.predict(X_dset)
    
    mpreds = None
    with torch.no_grad():
        probs = hfuncmodel(X_dset)
        mpreds = (probs > hthresh).int().flatten()

    mmpreds = np.array(mpreds, dtype=int).flatten()    
    merror = np.mean(np.abs(mmpreds - y_fstar))
    
    mfnr, mfpr = compute_fnr_fpr(y_fstar, mmpreds)

    return merror, mfnr, mfpr



def compute_improv_error_rates(hfunc, fstarm, hthreshold, hlossfunc, X_subset, y_subset, i_features, i_eps, i_alpha, i_iters, i_fn):
    
    with torch.no_grad():
        ypredictions = (hfunc(X_subset) > hthreshold).int().flatten()
    
    who_improves = []
    if not i_fn:
        who_improves = torch.where(ypredictions == 0)[0]
    else:
        ytrue_sub = y_subset if isinstance(y_subset, torch.Tensor) else torch.tensor(y_subset)
        who_improves = torch.where((ypredictions == 0) & (torch.tensor(ytrue_sub) == 1))[0]
        
    
    X_neg = X_subset[who_improves]
    y_neg = y_subset[who_improves]
    
    
    ###########
    adv_X = pgd_improvement(hfunck=hfunc, 
                            loss_funck=hlossfunc, 
                            inputsk=X_neg, 
                            labelsk=y_neg, 
                            sel_feat_idsk=i_features, 
                            epsilonbk=i_eps, 
                            alpha_valk=i_alpha, 
                            num_itersk=i_iters).detach()
    
    
    pred_before = []
    pred_after = []
    ###########
    with torch.no_grad():
        pred_before = (hfunc(X_neg) > hthreshold).int().flatten()
        pred_after = (hfunc(adv_X) > hthreshold).int().flatten()
        
    
    ###########
    final_X = []
    for orig, adv, before, after in zip(X_neg, adv_X, pred_before, pred_after):
        if before == after: 
            final_X.append(orig)
        else:
            final_X.append(adv)
    finalX = torch.stack(final_X)
    
    
    
    ###########
    X_subset_adv = X_subset.clone()
    X_subset_adv[who_improves] = finalX
    
    
    ###########
    errorR, fnrR, fprR = calculate_error_rate(hfuncmodel=hfunc, 
                                            fstarx=fstarm,
                                            X_dset=X_subset_adv, 
                                            hthresh=hthreshold)
    
        
    return errorR, fnrR, fprR



In [ ]:
def general_line_plot(trends_list_plt, figw_plt, figh_plt, legh_plt, legw_plt, legfont_plt, num_cols_plt, ylabel_plt, color_map_plt, save_as_plt):
    
    plt.figure(figsize=(figw_plt, figh_plt))

    for trend_plt in trends_list_plt:
        th_plt = trend_plt['threshold']
        fn_plt = trend_plt['flag']
        ibudgets_plt = trend_plt['ibudgets']
        errors_ls_plt = trend_plt['errors_ls']


        if fn_plt == True:
            fns_plt = '$\{\mathbf{x} \ | \ h(\mathbf{x})=-1, f^\star(\mathbf{x})=1\}$'
        else:
            fns_plt = '$\{\mathbf{x} \ | \ h(\mathbf{x})=-1\}$'
            
        base_color_plt = color_map_plt[th_plt]
        linestyle_plt = '-' if fn_plt else '--'
        label_plt = f"threshold={th_plt}"


        plt.plot(ibudgets_plt, errors_ls_plt, 
                 marker='o', linestyle=linestyle_plt, color=base_color_plt, linewidth=5, markersize=8, label=label_plt, alpha=0.7)


    min_errors_plt = [(trend_plt['ibudgets'][trend_plt['errors_ls'].index(min(trend_plt['errors_ls']))], 
                       min(trend_plt['errors_ls'])) for trend_plt in trends_list_plt]
    for ib_plt, er_plt in min_errors_plt:
        plt.scatter([ib_plt], [er_plt], color='red', s=100, zorder=5)


    plt.xlabel('Improvement budget ($r$)', fontsize=22)
    plt.ylabel(ylabel_plt, fontsize=22)
    plt.grid(color='gray', linestyle='--', alpha=0.7)
    plt.legend(fontsize=legfont_plt,            
                loc='upper center',     
                bbox_to_anchor=(legh_plt, legw_plt),  
                ncol=num_cols_plt,                 
                frameon=True          
            )

    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)
    plt.tight_layout()
    plt.savefig(save_as_plt, dpi=500, bbox_inches='tight')
    # plt.show()
    plt.close()


    
    
def draw_heatmap(error_ratei,  minval, maxval, ylabel, save_as):
    
    error_ratei = np.array(error_ratei)
    plt.figure(figsize=(12, 5))  
    ax = sns.heatmap(
        error_ratei, 
        annot=True, 
        fmt=".3f", 
        cmap="Spectral", 
        linewidth=.5,
        xticklabels=ibudgets, 
        yticklabels=thresholds, 
        annot_kws={"size": 12},  
        cbar_kws={"label": ylabel},
        vmin=minval,
        vmax=maxval
    )


    colorbar = ax.collections[0].colorbar
    colorbar.set_label(ylabel, fontsize=16)  
    colorbar.ax.tick_params(labelsize=14) 


    plt.xlabel("Improvement Budget ($r$)", fontsize=16, labelpad=10)
    plt.ylabel("Threshold Values", fontsize=16, labelpad=10)
    plt.xticks(fontsize=14)  
    plt.yticks(fontsize=14)

    plt.tight_layout(pad=2.0)
    plt.savefig(save_as, dpi=500)
    # plt.show()
    plt.close()
    
  

In [ ]:
dataset_name = 'oulad'

x_traindf, x_testdf, y_traindf, y_testdf = read_preprocessed_data(data_id=dataset_name, 
                                                datapath=proc_datapath)


####################
Xtr = np.array(x_traindf)
ytr = np.array(y_traindf)
Xts = np.array(x_testdf)
yts = np.array(y_testdf)


scaler = StandardScaler()
X_combined = np.vstack((Xtr, Xts))
y = np.vstack((ytr, yts)).ravel()
Xprox = scaler.fit_transform(X_combined)


Xtr1 = Xprox[:len(Xtr)]
Xts1 = Xprox[len(Xtr):]

Xtrx = torch.tensor(Xtr1, dtype=torch.float32)
Xtsx = torch.tensor(Xts1, dtype=torch.float32)
ytrx = torch.tensor(ytr, dtype=torch.long)
ytsx = torch.tensor(yts, dtype=torch.long) 

ytrx = ytrx.flatten()  
ytsx = ytsx.flatten() 
    
train_loader = DataLoader(TensorDataset(Xtrx, ytrx), batch_size=64, shuffle=False)
test_loader = DataLoader(TensorDataset(Xtsx, ytsx), batch_size=64, shuffle=False)


In [ ]:
y.shape, Xprox.shape

In [ ]:
main_folder = './'+dataset_name+'/'
storing_folder = './'+dataset_name+'/models/'
figure_folder = main_folder+'linf/'

for folder in [main_folder, storing_folder, figure_folder]:
    os.makedirs(folder, exist_ok=True)
    


In [ ]:
if dataset_name == 'adult':
    ##### Specify feature indices for improvement
    feature_names = ['fnlwgt', 'education', 'education-num', 'occupation', 'capital-gain', 'capital-loss', 'hours-per-week']
    feature_names_all = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 
                         'occupation', 'relationship', 'race', 'gender', 'capital-gain', 'capital-loss', 
                         'hours-per-week', 'native-country']

if dataset_name == 'law':
    ##### Specify feature indices for improvement
    feature_names = ['decile1b', 'decile3', 'lsat', 'ugpa', 'zfygpa', 'zgpa', 'fulltime']
    feature_names_all = ['decile1b', 'decile3', 'lsat', 'ugpa', 'zfygpa', 'zgpa', 'fulltime',
                           'fam_inc', 'male', 'tier', 'race']        

if dataset_name == 'oulad':
    ##### Specify feature indices for improvement
    feature_names = ['code_module', 'code_presentation', 'highest_education', 'imd_band', 'studied_credits']
    feature_names_all = ['code_module', 'code_presentation', 'id_student', 'gender', 'fstar_modelion', 'highest_education', 'imd_band', 
                           'age_band', 'num_of_prev_attempts','studied_credits', 'disability']      
    
if dataset_name == 'synthetic2':
    ##### Specify feature indices for improvement
    feature_names = ['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7']  
    feature_names_all = ['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7']   
    

feature_map = {name: i for i, name in enumerate(feature_names_all)}  
feature_indices_def = [feature_map[name] for name in feature_names]

num_features = Xtr.shape[1]
num_features

In [ ]:
x_traindf

In [ ]:
fstar_model = DecisionTreeClassifier(random_state=42)

fstar_model.fit(Xprox, y)
print(f'evaluate score: {fstar_model.score(Xprox, y)}')

ypre = fstar_model.predict(Xprox)
acc = accuracy_score(y, ypre)
print(f"Accuracy: {acc:.4f}")

ypr = fstar_model.predict(Xtrx.numpy())
ac = accuracy_score(ytr.ravel(), ypr)
print(f"Accuracy: {ac:.4f}")


## BCE AND WBCELossx

In [ ]:
######################
for c_id in [0, 1]:
    
    if c_id == 0:
        set_random_seed(42)
        lsmodel = NNClassifier(input_dim=num_features)
        optimizer = optim.Adam(lsmodel.parameters(), lr=0.001)
        criterion = BCELossx() 

        for epoch in range(100):
            lsmodel.train()
            for batch_x, batch_y in train_loader:
                optimizer.zero_grad()
                outputs = lsmodel(batch_x)
                batch_y = batch_y.view(-1, 1).float() 
                loss = criterion(outputs, batch_y)  
                loss.backward()
                optimizer.step()

        torch.save(lsmodel.state_dict(), f'{storing_folder}/bcemodel_linf.pth')
        
        ##################
        ibudgets = [0.0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.7, 1.0, 2.0, 4.0]
        thresholds = [0.5, 0.7, 0.8, 0.9]
        color_map = {
            0.5: '#1f77b4',  
            0.7: '#2ca02c',  
            0.8: '#ff7f0e',  
            0.9: '#9467bd',  
        }
        lsfunc = 'bce'


        
    else:
        #################
        set_random_seed(42)
        lsmodel = NNClassifier(input_dim=num_features)
        optimizer = optim.Adam(lsmodel.parameters(), lr=0.001)
        criterion = WBCELossx()

        for epoch in range(100):
            lsmodel.train()
            for batch_x, batch_y in train_loader:
                optimizer.zero_grad()
                outputs = lsmodel(batch_x)
                batch_y = batch_y.view(-1, 1).float() 
                loss = criterion(outputs, batch_y)  
                loss.backward()
                optimizer.step()

        torch.save(lsmodel.state_dict(), f'{storing_folder}/pfpmodel_linf.pth')


        ##################
        ibudgets = [0.0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.7, 1.0, 2.0, 4.0]
        thresholds = [0.5, 0.7, 0.8, 0.9]
        color_map = {
            0.5: '#1f77b4',  
            0.7: '#2ca02c',  
            0.8: '#ff7f0e',  
            0.9: '#9467bd',  
        }
        lsfunc = 'fpl'

        
        

        
    ##################    
    ##################    
    ##################
    alpha = 0.01
    iters = 500
 
    all_trends = []
    error_ratez = []
    fntrue_trends = []
    fnfalse_trends = []
    fntrue_error_ratez = []
    fnfalse_error_ratez = []
    
    
    fnr_all_trends = []
    fnr_error_ratez = []
    fnr_fntrue_trends = []
    fnr_fnfalse_trends = []
    fnr_fntrue_error_ratez = []
    fnr_fnfalse_error_ratez = []
    
    
    fpr_all_trends = []
    fpr_error_ratez = []
    fpr_fntrue_trends = []
    fpr_fnfalse_trends = []
    fpr_fntrue_error_ratez = []
    fpr_fnfalse_error_ratez = []


    ##################
    for th in thresholds:

        errors_ls = []
        fnr_errors_ls = []
        fpr_errors_ls = []
        fnl = False
        for r in ibudgets:
            error_ratex, fnr_ratex, fpr_ratex = compute_improv_error_rates(
                                                                    hfunc=lsmodel, 
                                                                    fstarm=fstar_model, 
                                                                    hthreshold=th, 
                                                                    hlossfunc=criterion, 
                                                                    X_subset=Xtsx, 
                                                                    y_subset=ytsx, 
                                                                    i_features=feature_indices_def, 
                                                                    i_eps=r, 
                                                                    i_alpha=alpha, 
                                                                    i_iters=iters, 
                                                                    i_fn=fnl
                                                                )
            errors_ls.append(error_ratex)
            fnr_errors_ls.append(fnr_ratex)
            fpr_errors_ls.append(fpr_ratex)


        ###########
        ###########
        all_trends.append({'threshold': th, 'flag': fnl, 'ibudgets': ibudgets, 'errors_ls': errors_ls})
        ###
        fnr_all_trends.append({'threshold': th, 'flag': fnl, 'ibudgets': ibudgets, 'errors_ls': fnr_errors_ls})
        ###
        fpr_all_trends.append({'threshold': th, 'flag': fnl, 'ibudgets': ibudgets, 'errors_ls': fpr_errors_ls})
        
        if not fnl:
            fnfalse_trends.append({'threshold': th, 'flag': fnl, 'ibudgets': ibudgets, 'errors_ls': errors_ls})
            fnfalse_error_ratez.append(errors_ls)
            ###
            fnr_fnfalse_trends.append({'threshold': th, 'flag': fnl, 'ibudgets': ibudgets, 'errors_ls': fnr_errors_ls})
            fnr_fnfalse_error_ratez.append(fnr_errors_ls)
            ###
            fpr_fnfalse_trends.append({'threshold': th, 'flag': fnl, 'ibudgets': ibudgets, 'errors_ls': fpr_errors_ls})
            fpr_fnfalse_error_ratez.append(fpr_errors_ls)
        else:
            fntrue_trends.append({'threshold': th, 'flag': fnl, 'ibudgets': ibudgets, 'errors_ls': errors_ls})
            fntrue_error_ratez.append(errors_ls)
            ###
            fnr_fntrue_trends.append({'threshold': th, 'flag': fnl, 'ibudgets': ibudgets, 'errors_ls': fnr_errors_ls})
            fnr_fntrue_error_ratez.append(fnr_errors_ls)
            ###
            fpr_fntrue_trends.append({'threshold': th, 'flag': fnl, 'ibudgets': ibudgets, 'errors_ls': fpr_errors_ls})
            fpr_fntrue_error_ratez.append(fpr_errors_ls)                


    ##################
    ##################
    if len(fnfalse_trends) > 0:
        general_line_plot(trends_list_plt=fnfalse_trends, 
                          figw_plt=10, 
                          figh_plt=6, 
                          legh_plt=0.5, 
                          legw_plt=1.32, 
                          legfont_plt=20, 
                          num_cols_plt=2,
                          ylabel_plt='Model ($h$) Error',
                          color_map_plt=color_map,
                          save_as_plt=figure_folder+'fnfalse_trends_lineplot_'+str(lsfunc)+'_linf_oulad.pdf')

        draw_heatmap(error_ratei=fnfalse_error_ratez, 
                     minval=0.01, 
                     maxval=0.98, 
                     ylabel="Model ($h$) Error",
                     save_as=figure_folder+'fnfalse_trends_heatmap_'+str(lsfunc)+'_linf_oulad.pdf')
        ######
        general_line_plot(trends_list_plt=fnr_fnfalse_trends, 
                  figw_plt=10, 
                  figh_plt=6, 
                  legh_plt=0.5, 
                  legw_plt=1.32, 
                  legfont_plt=20, 
                  num_cols_plt=2,
                  ylabel_plt='Model ($h$) FNR',
                  color_map_plt=color_map,
                  save_as_plt=figure_folder+'fnr_fnfalse_trends_lineplot_'+str(lsfunc)+'_linf_oulad.pdf')

        draw_heatmap(error_ratei=fnr_fnfalse_error_ratez, 
                     minval=0.01, 
                     maxval=0.98,  
                     ylabel='Model ($h$) FNR', 
                     save_as=figure_folder+'fnr_fnfalse_trends_heatmap_'+str(lsfunc)+'_linf_oulad.pdf')
        
        ######
        general_line_plot(trends_list_plt=fpr_fnfalse_trends, 
                  figw_plt=10, 
                  figh_plt=6, 
                  legh_plt=0.5, 
                  legw_plt=1.32, 
                  legfont_plt=20, 
                  num_cols_plt=2,
                  ylabel_plt='Model ($h$) FPR',
                  color_map_plt=color_map,
                  save_as_plt=figure_folder+'fpr_fnfalse_trends_lineplot_'+str(lsfunc)+'_linf_oulad.pdf')

        draw_heatmap(error_ratei=fpr_fnfalse_error_ratez, 
                     minval=0.01, 
                     maxval=0.98,  
                     ylabel='Model ($h$) FPR', 
                     save_as=figure_folder+'fpr_fnfalse_trends_heatmap_'+str(lsfunc)+'_linf_oulad.pdf')

    ##################
    ##################
    if len(fntrue_trends) > 0:
        general_line_plot(trends_list_plt=fntrue_trends, 
                          figw_plt=10, 
                          figh_plt=6, 
                          legh_plt=0.5, 
                          legw_plt=1.32, 
                          legfont_plt=20,
                          num_cols_plt=2, 
                          ylabel_plt='Model ($h$) Error',   
                          color_map_plt=color_map,
                          save_as_plt=figure_folder+'fntrue_trends_lineplot_'+str(lsfunc)+'_linf_oulad.pdf')

        draw_heatmap(error_ratei=fntrue_error_ratez, 
                     minval=0.01, 
                     maxval=0.98, 
                     ylabel='Model ($h$) Error',
                     save_as=figure_folder+'fntrue_trends_heatmap_'+str(lsfunc)+'_linf_oulad.pdf')
        
        
        ######
        general_line_plot(trends_list_plt=fnr_fntrue_trends, 
                  figw_plt=10, 
                  figh_plt=6, 
                  legh_plt=0.5, 
                  legw_plt=1.32, 
                  legfont_plt=20,
                  num_cols_plt=2, 
                  ylabel_plt='Model ($h$) FNR',
                  color_map_plt=color_map,
                  save_as_plt=figure_folder+'fnr_fntrue_trends_lineplot_'+str(lsfunc)+'_linf_oulad.pdf')

        draw_heatmap(error_ratei=fnr_fntrue_error_ratez, 
                     minval=0.01, 
                     maxval=0.98, 
                     ylabel='Model ($h$) FNR',
                     save_as=figure_folder+'fnr_fntrue_trends_heatmap_'+str(lsfunc)+'_linf_oulad.pdf')

        ######
        general_line_plot(trends_list_plt=fpr_fntrue_trends, 
                  figw_plt=10, 
                  figh_plt=6, 
                  legh_plt=0.5, 
                  legw_plt=1.32, 
                  legfont_plt=20,
                  num_cols_plt=2,  
                  ylabel_plt='Model ($h$) FPR',
                  color_map_plt=color_map,
                  save_as_plt=figure_folder+'fpr_fntrue_trends_lineplot_'+str(lsfunc)+'_linf_oulad.pdf')

        draw_heatmap(error_ratei=fpr_fntrue_error_ratez, 
                     minval=0.01, 
                     maxval=0.98, 
                     ylabel='Model ($h$) FPR',
                     save_as=figure_folder+'fpr_fntrue_trends_heatmap_'+str(lsfunc)+'_linf_oulad.pdf')





    ##################
    general_line_plot(trends_list_plt=all_trends, 
                      figw_plt=10, 
                      figh_plt=6, 
                      legh_plt=0.5, 
                      legw_plt=1.32, 
                      legfont_plt=20, 
                      num_cols_plt=2, 
                      ylabel_plt='Model ($h$) Error',
                      color_map_plt=color_map,
                      save_as_plt=figure_folder+'all_trends_lineplot_'+str(lsfunc)+'_linf_oulad.pdf' )


    #####
    general_line_plot(trends_list_plt=fnr_all_trends, 
                  figw_plt=10, 
                  figh_plt=6, 
                  legh_plt=0.5, 
                  legw_plt=1.32, 
                  legfont_plt=20, 
                  num_cols_plt=2, 
                  ylabel_plt='Model ($h$) FNR',
                  color_map_plt=color_map,
                  save_as_plt=figure_folder+'fnr_all_trends_lineplot_'+str(lsfunc)+'_linf_oulad.pdf' )
    #####
    general_line_plot(trends_list_plt=fpr_all_trends, 
                  figw_plt=10, 
                  figh_plt=6, 
                  legh_plt=0.5, 
                  legw_plt=1.32, 
                  legfont_plt=20, 
                  num_cols_plt=2,  
                  ylabel_plt='Model ($h$) FPR',
                  color_map_plt=color_map,
                  save_as_plt=figure_folder+'fpr_all_trends_lineplot_'+str(lsfunc)+'_linf_oulad.pdf' )




## Only w

In [ ]:
#################    
#################

for wfpidx, wfp in enumerate([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]):
    
    set_random_seed(42)
    lsmodelz = NNClassifier(input_dim=num_features)
    optimizerz = optim.Adam(lsmodelz.parameters(), lr=0.001)
    criterionz = WBCELossx(false_positive_weight=wfp)

    for epoch in range(100):
        lsmodelz.train()
        for batch_x, batch_y in train_loader:
            optimizerz.zero_grad()
            outputs = lsmodelz(batch_x)
            batch_y = batch_y.view(-1, 1).float() 
            loss = criterionz(outputs, batch_y)  
            loss.backward()
            optimizerz.step()

    torch.save(lsmodelz.state_dict(), f'{storing_folder}/pfpmodel_'+str(wfp)+'_linf.pth')


    ##################
    ibudgets = [0.0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.7, 1.0, 2.0, 4.0]
    thresholdsx = [0.5, 0.7, 0.8, 0.9]
    color_map = {
        0.5: '#1f77b4',  
        0.7: '#2ca02c',  
        0.8: '#ff7f0e',  
        0.9: '#9467bd',  
    }
    lsfunc = 'fpl'


    ##################    
    ##################    
    ##################
    alphaz = 0.01
    itersz = 500

    all_trendsz = []

    ##################
    for th in thresholdsx:
        fnz = False

        errors_lsz = []
        for r in ibudgets:
            error_ratexz, fnr_ratexz, fpr_ratexz = compute_improv_error_rates(
                                                                    hfunc=lsmodelz, 
                                                                    fstarm=fstar_model, 
                                                                    hthreshold=th, 
                                                                    hlossfunc=criterionz, 
                                                                    X_subset=Xtsx, 
                                                                    y_subset=ytsx, 
                                                                    i_features=feature_indices_def, 
                                                                    i_eps=r, 
                                                                    i_alpha=alphaz, 
                                                                    i_iters=itersz, 
                                                                    i_fn=fnz
                                                                )
            errors_lsz.append(error_ratexz)

        ###########
        all_trendsz.append({'threshold': th, 'flag': fnz, 'ibudgets': ibudgets, 'errors_ls': errors_lsz})

        
    
    ##################
    general_line_plot(trends_list_plt=all_trendsz, 
                      figw_plt=10, 
                      figh_plt=6, 
                      legh_plt=0.5, 
                      legw_plt=1.32, 
                      legfont_plt=20, 
                      num_cols_plt=2, 
                      ylabel_plt='Model ($h$) Error',
                      color_map_plt=color_map,
                      save_as_plt=figure_folder+'all_trends_lineplot_'+str(lsfunc)+'_'+str(wfp)+'_linf_oulad.pdf' )


    


## BOTH

In [ ]:
######################
ibudgets_set = [
    [0.0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.7, 1.0, 2.0, 4.0]
]
figname = ['lfc']

fns = ['$\{\mathbf{x}: h(\mathbf{x})=0, f^\star(\mathbf{x})=1\}$', '$\{\mathbf{x}: h(\mathbf{x})=0\}$']
lossfunc = ['$\mathcal{L}_{BCE}$', '$\mathcal{L}_{wBCE}$']

colors = ['#4CAF50', '#FF5722', '#2196F3', '#FFC107']
linestyles = ['-', '--']
markers = ['o', 's', 'D', '^']

for fig_idx, ibudgets in enumerate(ibudgets_set):

    plt.figure(figsize=(10, 5))
    ########
    fnrx_errors_ls_all = []
    fprx_errors_ls_all = []
        
    ################
    for cidx in [0, 1]:
        if cidx == 0:
            set_random_seed(42)
            criterion = BCELossx()
            genmodel = NNClassifier(input_dim=num_features)
            genmodel.load_state_dict(torch.load(f'{storing_folder}/bcemodel_linf.pth'))
        else:
            set_random_seed(42)
            criterion = WBCELossx()
            genmodel = NNClassifier(input_dim=num_features)
            genmodel.load_state_dict(torch.load(f'{storing_folder}/pfpmodel_linf.pth'))
        
        alpha = 0.01
        iters = 500
        th = 0.5
        
        #########
        sub_fnrx_errors_ls_all = []
        sub_fprx_errors_ls_all = []
        
        #########
        fn_idx = 0 
        fn = False
        errorsx_lsxx = []
        fnrx_errors_lsxx = []
        fprx_errors_lsxx = []
        for r in ibudgets:
            error_rate1xx, fnr_rate1xx, fpr_rate1xx = compute_improv_error_rates(
                                                                    hfunc=genmodel,
                                                                    fstarm=fstar_model,
                                                                    hthreshold=th,
                                                                    hlossfunc=criterion,
                                                                    X_subset=Xtsx,
                                                                    y_subset=ytsx,
                                                                    i_features=feature_indices_def,
                                                                    i_eps=r,
                                                                    i_alpha=alpha,
                                                                    i_iters=iters,
                                                                    i_fn=fn
                                                                )
            errorsx_lsxx.append(error_rate1xx)
            fnrx_errors_lsxx.append(fnr_rate1xx)
            fprx_errors_lsxx.append(fpr_rate1xx)


        #####      
        sub_fnrx_errors_ls_all.append(fnrx_errors_lsxx)
        sub_fprx_errors_ls_all.append(fprx_errors_lsxx)
        
        


        ####
        label = f'{lossfunc[cidx]}'
        color = colors[cidx]
        linestyle = linestyles[cidx]
        marker = markers[cidx]

        ######
        plt.plot(ibudgets, errorsx_lsxx, marker=marker, linestyle=linestyle, 
                 color=color, linewidth=5, markersize=8, label=label)
        
        #####
        fnrx_errors_ls_all.append(sub_fnrx_errors_ls_all)
        fprx_errors_ls_all.append(sub_fprx_errors_ls_all)  
    
    
    ###### 
    ###### 
    plt.xlabel('Improvement Budget ($r$)', fontsize=24)
    plt.ylabel('Model ($h$) Error', fontsize=24)
    plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)
    plt.xticks(fontsize=23)
    plt.yticks(fontsize=23)
    plt.legend(fontsize=27,  
                labelcolor='black',                   
                loc='upper center',     
                bbox_to_anchor=(0.5, 1.3),  
                ncol=2,
                frameon=True          
            )
    plt.tight_layout()
    plt.savefig(figure_folder + figname[fig_idx] + '_thahalf_error_droprate_linf_oulad.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    
    
    ###### FNR
    plt.figure(figsize=(10, 5))
    for c_idx in [0, 1]:
        fn_idxx = 0
        label = f'{lossfunc[c_idx]}'
        color = colors[c_idx]
        linestyle = linestyles[c_idx]
        marker = markers[c_idx]

        ######
        fnrx_errors_lsa = fnrx_errors_ls_all[c_idx][fn_idxx]
        plt.plot(ibudgets, fnrx_errors_lsa, marker=marker, linestyle=linestyle, 
                 color=color, linewidth=5, markersize=8, label=label)
    
    plt.xlabel('Improvement Budget ($r$)', fontsize=24)
    plt.ylabel('Model ($h$) FNR', fontsize=24)
    plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)
    plt.xticks(fontsize=23)
    plt.yticks(fontsize=23)
    plt.legend(fontsize=27,  
                labelcolor='black',                   
                loc='upper center',     
                bbox_to_anchor=(0.5, 1.3),  
                ncol=2,
                frameon=True          
            )
    plt.tight_layout()
    plt.savefig(figure_folder + figname[fig_idx] + '_thahalf_FNR_droprate_linf_oulad.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    
    
    ###### FPR
    plt.figure(figsize=(10, 5))
    for c_idz in [0, 1]:
        fn_idxz = 0
        label = f'{lossfunc[c_idz]}'
        color = colors[c_idz]
        linestyle = linestyles[c_idz]
        marker = markers[c_idz]

        #####
        fprx_errors_lsa = fprx_errors_ls_all[c_idz][fn_idxz]
        plt.plot(ibudgets, fprx_errors_lsa, marker=marker, linestyle=linestyle, 
                 color=color, linewidth=5, markersize=8, label=label)
    
    plt.xlabel('Improvement Budget ($r$)', fontsize=24)
    plt.ylabel('Model ($h$) FPR', fontsize=24)
    plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)
    plt.xticks(fontsize=23)
    plt.yticks(fontsize=23)
    plt.legend(fontsize=27,  
                labelcolor='black',                   
                loc='upper center',     
                bbox_to_anchor=(0.5, 1.3),  
                ncol=2,
                frameon=True          
            )
    plt.tight_layout()
    plt.savefig(figure_folder + figname[fig_idx] + '_thahalf_FPR_droprate_linf_oulad.pdf', dpi=300, bbox_inches='tight')
    plt.show()


## Only w

In [ ]:
ibudgets_set = [
    [0.0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.7, 1.0, 2.0, 4.0]
]
figname = ['lfconlyw']


colors = ['#000000', '#FFEB3B', '#D50000', '#8E24AA', '#43A047', '#FF5722', '#F57C00', '#7B1FA2', '#0288D1', '#2E7D32']
linestyles = ['-', '--', '-.', ':']
markers = ['o', 's', 'D', '^', 'v', 'p', 'h', 'x', '+', '*']

wall = [0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]

line_properties = [
    {'color': colors[i % len(colors)], 'linestyle': linestyles[i % len(linestyles)], 'marker': markers[i % len(markers)], 'alpha': 0.6, 'linewidth': 5, 'markersize': 10}
    for i in range(len(wall))
]

for th in [0.5, 0.9, 0.99]:

    set_random_seed(42)
    
    for fig_idx, ibudgets in enumerate(ibudgets_set):

        plt.figure(figsize=(14, 8))

        for idx, wfpo in enumerate(wall):

            if wfpo == 0:
                criterion = BCELossx()
                genmodel = NNClassifier(input_dim=num_features)
                genmodel.load_state_dict(torch.load(f'{storing_folder}/bcemodel_linf.pth'))
            else:
                criterion = WBCELossx(false_positive_weight=wfpo)
                genmodel = NNClassifier(input_dim=num_features)
                genmodel.load_state_dict(torch.load(f'{storing_folder}/pfpmodel_'+str(wfpo)+'_linf.pth'))

            
            ##########
            alpha = 0.01
            iters = 500

            errorsx_lsy = []
            for r in ibudgets:
                error_rate1xy, fnr_rate1xy, fpr_rate1xy = compute_improv_error_rates(
                                                                        hfunc=genmodel,
                                                                        fstarm=fstar_model,
                                                                        hthreshold=th,
                                                                        hlossfunc=criterion,
                                                                        X_subset=Xtsx,
                                                                        y_subset=ytsx,
                                                                        i_features=feature_indices_def,
                                                                        i_eps=r,
                                                                        i_alpha=alpha,
                                                                        i_iters=iters,
                                                                        i_fn=False
                                                                    )
                errorsx_lsy.append(error_rate1xy)


            if wfpo == 0:
                label = r'$\mathcal{L}_{BCE}$'
            else:
                label = r'$\mathcal{L}_{wBCE}, (w_{FP}=' + str(wfpo) + ')$'


            line_style = line_properties[idx]

            plt.plot(ibudgets, errorsx_lsy, 
                     marker=line_style['marker'], 
                     linestyle=line_style['linestyle'], 
                     color=line_style['color'], 
                     alpha=line_style['alpha'], 
                     linewidth=line_style['linewidth'], 
                     markersize=line_style['markersize'], 
                     label=label)

        plt.xlabel('Improvement Budget ($r$)', fontsize=26)
        plt.ylabel('Model ($h$) Error', fontsize=26)
        plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)
        plt.xticks(fontsize=23)
        plt.yticks(fontsize=23)
        font_properties = font_manager.FontProperties(weight='semibold', size=24)
        plt.legend(fontsize=24,  
                    prop=font_properties,
                    labelcolor='black',                   
                    loc='upper center',     
                    bbox_to_anchor=(0.5, 1.41),  
                    ncol=3,
                    frameon=True,  
                    columnspacing=0.77,
                    labelspacing=0.44  
        )
        plt.tight_layout()
        plt.savefig(figure_folder + figname[fig_idx] + '_'+str(th)+'_error_droprate_linfx_oulad.pdf', dpi=600, bbox_inches='tight')
        plt.show()

        